# Environment check
Run these cells top-to-bottom after `docker compose up -d` to confirm HDFS, Spark, and Kafka are all reachable from this notebook.

In [1]:
# 1. Spark connection (standalone cluster, not local mode)
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EnvCheck")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/opt/conda/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2707b794-3939-4606-b352-3efab8a39036;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.0 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloa

26/09/09 19:59:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/09 19:59:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


26/09/09 21:08:06 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/09/09 21:08:06 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/09/09 21:08:06 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/09/09 21:08:18 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.10: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.


In [ ]:
# 2. HDFS connection
df = spark.createDataFrame([(1, "hello"), (2, "hdfs")], ["id", "msg"])
df.write.mode("overwrite").csv("hdfs://namenode:9000/labs/envcheck")
spark.read.csv("hdfs://namenode:9000/labs/envcheck").show()

In [ ]:
# 3. Kafka connection (produce + consume one message)
!pip install -q kafka-python
from kafka import KafkaProducer, KafkaConsumer
import json, time

producer = KafkaProducer(bootstrap_servers="kafka:9092", value_serializer=lambda v: json.dumps(v).encode())
producer.send("envcheck", {"status": "ok"})
producer.flush()

consumer = KafkaConsumer("envcheck", bootstrap_servers="kafka:9092", auto_offset_reset="earliest", consumer_timeout_ms=5000)
for msg in consumer:
    print(msg.value)
print("Kafka round-trip OK")

In [ ]:
spark.stop()